In [2]:
import pandas as pd

datasets_dir = "./data"

btt25_ds = pd.read_csv(f'{datasets_dir}/btt25_sentence_ph_trainset.csv')
owt2_ds = pd.read_parquet(f'{datasets_dir}/owt2_sentence_ph_trainset.parquet')
hvd_ds = pd.read_parquet(f'{datasets_dir}/hwd_sentence_ph_trainset.parquet')
switchboard_ds = pd.read_parquet(f'{datasets_dir}/switchboard_sentence_ph_trainset.parquet')

print("Loaded All 4 Datasets with:")
print(f'BTT25: {len(btt25_ds)} samples')
print(f'OWT2: {len(owt2_ds)} samples')
print(f'HVD: {len(hvd_ds)} samples')
print(f'Switchboard: {len(switchboard_ds)} samples')

Loaded All 4 Datasets with:
BTT25: 8072 samples
OWT2: 1113734 samples
HVD: 720 samples
Switchboard: 157554 samples


In [3]:
switchboard_ds['split'].unique()

array(['train', 'eval', 'test'], dtype=object)

### Config

In [4]:
from transformers import (
    GPT2LMHeadModel, 
    GPT2Tokenizer, 
    AdamW, 
    get_linear_schedule_with_warmup
)

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

/mnt/task_runtime/brain-to-text-25/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

In [5]:
from models.phoneme_to_text.config import (
    PHONEME_TOKENS, SPECIAL_TOKENS, TRAIN_DATA_PATH
)
from models.phoneme_to_text.dataset import PhonemeTextDataset

tokenizer.add_special_tokens({
    "pad_token": SPECIAL_TOKENS["pad_token"],
    "sep_token": SPECIAL_TOKENS["sep_token"],
    "bos_token": SPECIAL_TOKENS["bos_token"],
    "eos_token": SPECIAL_TOKENS["eos_token"],
    "additional_special_tokens": PHONEME_TOKENS
})

loader = PhonemeTextDataset(data_dir=TRAIN_DATA_PATH, tokenizer=tokenizer)
raw_datasets = loader.load_and_prepare_datasets()

Loading datasets from ./data...


Generating train split: 0 examples [00:00, ? examples/s]

Parsing stringified lists (num_proc=4):   0%|          | 0/8072 [00:00<?, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/157554 [00:00<?, ? examples/s]

Filter:   0%|          | 0/157554 [00:00<?, ? examples/s]

Filter:   0%|          | 0/157554 [00:00<?, ? examples/s]

Merging datasets...
Final Counts -> Train: 1223469, Val: 24601, Test: 32010


In [1]:
import torch
import numpy as np
from transformers import GPT2Tokenizer
from tqdm import tqdm
import matplotlib.pyplot as plt
import os

from config import (
    TRAIN_DATA_PATH, 
    SPECIAL_TOKENS, 
    PHONEME_TOKENS
)
from models.phoneme_to_text.dataset import PhonemeTextDataset

def analyze_lengths():
    print("Initializing Tokenizer...")
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    
    # We must replicate the exact token setup to get accurate lengths
    tokenizer.add_special_tokens({
        "pad_token": SPECIAL_TOKENS["pad_token"],
        "sep_token": SPECIAL_TOKENS["sep_token"],
        "bos_token": SPECIAL_TOKENS["bos_token"],
        "eos_token": SPECIAL_TOKENS["eos_token"],
        "additional_special_tokens": PHONEME_TOKENS
    })

    print("Loading Dataset (Raw)...")
    # We use the loader to get the merged dataset, but we DO NOT call 
    # prepare_for_trainer because that function truncates the data!
    loader = PhonemeTextDataset(data_dir=TRAIN_DATA_PATH, tokenizer=tokenizer)
    dataset_dict = loader.load_and_prepare_datasets()
    
    # We only need to check the training set as it's the largest and most representative
    ds = dataset_dict['train']
    print(f"Analyzing {len(ds)} samples...")

    # Define a fast mapping function to compute lengths
    def compute_length(batch):
        lengths = []
        inputs = batch['input_phonemes']
        outputs = batch['output_sentence']
        
        for ph_seq, text in zip(inputs, outputs):
            # 1. Phoneme Length
            # Since we map 1 phoneme -> 1 special token <p:XX>, 
            # the length is simply the number of items in the list.
            p_len = len(ph_seq)
            
            # 2. Text Length
            # We must encode this to see how many BPE tokens GPT-2 uses
            t_len = len(tokenizer.encode(text, add_special_tokens=False))
            
            # 3. Special Tokens Length
            # <BOS> + [Phonemes] + <SEP> + [Text] + <EOS>
            # 1 + p_len + 1 + t_len + 1
            total_len = 1 + p_len + 1 + t_len + 1
            lengths.append(total_len)
            
        return {"length": lengths}

    # Run batched mapping (much faster than a for loop)
    results = ds.map(
        compute_length, 
        batched=True, 
        batch_size=1000, 
        remove_columns=ds.column_names, # We only keep the length column to save RAM
        desc="Calculating Lengths"
    )

    # Convert to numpy for stats
    lens = np.array(results['length'])

    # --- REPORT ---
    print("\n" + "="*30)
    print(" DATASET LENGTH STATISTICS ")
    print("="*30)
    print(f"Total Samples: {len(lens)}")
    print(f"Min Length:    {np.min(lens)}")
    print(f"Max Length:    {np.max(lens)}")
    print(f"Mean Length:   {np.mean(lens):.2f}")
    print(f"Median Length: {np.median(lens):.2f}")
    print("-" * 30)
    print("Percentiles:")
    print(f"90th %: {np.percentile(lens, 90):.2f}")
    print(f"95th %: {np.percentile(lens, 95):.2f}")
    print(f"99th %: {np.percentile(lens, 99):.2f}")
    print(f"99.9% : {np.percentile(lens, 99.9):.2f}")
    print("="*30)

    # Recommendation
    p999 = int(np.percentile(lens, 99.9))
    print(f"\nRECOMMENDATION: Set MAX_LENGTH to >= {p999}")
    
    # Optional: Histogram
    try:
        # Simple ASCII Histogram
        hist, bin_edges = np.histogram(lens, bins=10)
        print("\nDistribution:")
        for i in range(len(hist)):
            bar = "#" * int(hist[i] / np.max(hist) * 20)
            range_str = f"{int(bin_edges[i])}-{int(bin_edges[i+1])}"
            print(f"{range_str:<10} | {bar} ({hist[i]})")
    except:
        pass

analyze_lengths()

Initializing Tokenizer...


/mnt/task_runtime/brain-to-text-25/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading Dataset (Raw)...
Loading datasets from ./data...


Filter:   0%|          | 0/1113734 [00:00<?, ? examples/s]

Filter:   0%|          | 0/720 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/157554 [00:00<?, ? examples/s]

Filter:   0%|          | 0/157554 [00:00<?, ? examples/s]

Merging datasets...
Final Counts -> Train: 1222689, Val: 24593, Test: 32010
Analyzing 1222689 samples...


Calculating Lengths:   0%|          | 0/1222689 [00:00<?, ? examples/s]


 DATASET LENGTH STATISTICS 
Total Samples: 1222689
Min Length:    7
Max Length:    1104
Mean Length:   126.49
Median Length: 112.00
------------------------------
Percentiles:
90th %: 228.00
95th %: 272.00
99th %: 395.00
99.9% : 687.31

RECOMMENDATION: Set MAX_LENGTH to >= 687

Distribution:
7-116      | #################### (642287)
116-226    | ############## (455428)
226-336    | ### (100288)
336-445    |  (17244)
445-555    |  (4447)
555-665    |  (1580)
665-774    |  (735)
774-884    |  (411)
884-994    |  (235)
994-1104   |  (34)
